In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import scale
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

from sklearn.linear_model import RidgeCV
from sklearn.linear_model import LassoCV

import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

In [42]:
df = pd.read_csv(r"C:\Users\shaya\Downloads\SSDI\SSDI\DATA\Titanic-Dataset.csv", usecols=[1, 2, 5, 4, 6, 7, 9, 11],na_values="").dropna()

In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 712 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  712 non-null    int64  
 1   Pclass    712 non-null    int64  
 2   Gender    712 non-null    object 
 3   Age       712 non-null    float64
 4   SibSp     712 non-null    int64  
 5   Parch     712 non-null    int64  
 6   Fare      712 non-null    float64
 7   Embarked  712 non-null    object 
dtypes: float64(2), int64(4), object(2)
memory usage: 50.1+ KB


In [44]:
# Target is "Survived"

#cat: 'Pclass', 'Gender','SibSp', 'Parch','Embarked'
#cont: 'Age','Fare',

df.columns

Index(['Survived', 'Pclass', 'Gender', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked'],
      dtype='object')

In [45]:
# Standarizing Values

scaler = StandardScaler()
df[['Age','Fare']] = scaler.fit_transform(df[['Age','Fare']])

In [46]:
df = pd.get_dummies(df , columns=['Pclass', 'Gender','SibSp', 'Parch', 'Embarked'], drop_first=True)

In [47]:
df.columns

Index(['Survived', 'Age', 'Fare', 'Pclass_2', 'Pclass_3', 'Gender_male',
       'SibSp_1', 'SibSp_2', 'SibSp_3', 'SibSp_4', 'SibSp_5', 'Parch_1',
       'Parch_2', 'Parch_3', 'Parch_4', 'Parch_5', 'Parch_6', 'Embarked_Q',
       'Embarked_S'],
      dtype='object')

In [81]:
x = df[['Age','Fare']]
y = df["Survived"]

x = x.astype(float)
x = add_constant(x)

vif = pd.DataFrame()
vif["Features"] = x.columns
vif["vif"] = [variance_inflation_factor(x.values, i) for i in range(x.shape[1])]

vif

,Features,vif
0,const,1.000000
1,Age,1.008751
2,Fare,1.008751


In [82]:
x = df[['Age', 'Fare', 'Pclass_2', 'Pclass_3', 'Gender_male',
       'SibSp_1', 'SibSp_2', 'SibSp_3', 'SibSp_4', 'SibSp_5', 'Parch_1',
       'Parch_2', 'Parch_3', 'Parch_4', 'Parch_5', 'Parch_6', 'Embarked_Q',
       'Embarked_S']]


alphas = 10**np.linspace(10,-2,100)*0.5
ridge = Ridge()
coefs = []

for a in alphas:
    ridge.set_params(alpha = a)
    ridge.fit(x,y)
    coefs.append(ridge.coef_)
np.shape(coefs)

(100, 18)

In [83]:
lasso = Lasso()
coefs = []

for a in alphas:
    lasso.set_params(alpha = a)
    lasso.fit(x,y)
    coefs.append(lasso.coef_)
np.shape(coefs)

(100, 18)

In [84]:
ridgecv = RidgeCV (alphas = alphas)
ridgecv.fit(x,y)
bestalphaR = ridgecv.alpha_ #best alpha
print(bestalphaR)

3.0679536367065814


In [85]:
lassocv = LassoCV(alphas = alphas)
lassocv.fit(x,y)
bestalphaL = lassocv.alpha_ #best alpha
print(bestalphaL)

0.005


In [86]:
fitr = Ridge(alpha = bestalphaR)
fitr.fit(x,y)
fitr.coef_
print(pd.Series(fitr.coef_, index = x.columns))

Age           -0.081008
Fare           0.014151
Pclass_2      -0.163334
Pclass_3      -0.339073
Gender_male   -0.466482
SibSp_1       -0.000566
SibSp_2       -0.075799
SibSp_3       -0.243441
SibSp_4       -0.214237
SibSp_5       -0.203186
Parch_1        0.060598
Parch_2        0.024174
Parch_3        0.052067
Parch_4       -0.195984
Parch_5       -0.132137
Parch_6       -0.121152
Embarked_Q    -0.079197
Embarked_S    -0.059336
dtype: float64


In [87]:
fitl = Lasso (alpha = bestalphaL)
fitl.fit(x,y)
fitl.coef_
print(pd.Series(fitl.coef_, index = x.columns))

Age           -0.062751
Fare           0.018036
Pclass_2      -0.102876
Pclass_3      -0.310034
Gender_male   -0.457050
SibSp_1        0.000686
SibSp_2       -0.000000
SibSp_3       -0.000000
SibSp_4       -0.000000
SibSp_5       -0.000000
Parch_1        0.012349
Parch_2       -0.000000
Parch_3        0.000000
Parch_4       -0.000000
Parch_5       -0.000000
Parch_6       -0.000000
Embarked_Q    -0.000000
Embarked_S    -0.040536
dtype: float64


In [88]:
lm = smf.logit("Survived ~ Age + Pclass_2 + Pclass_3 + Gender_male", df).fit()
lm.summary()

Optimization terminated successfully.
         Current function value: 0.454137
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:               Survived   No. Observations:                  712
Model:                          Logit   Df Residuals:                      707
Method:                           MLE   Df Model:                            4
Date:                Fri, 08 May 2026   Pseudo R-squ.:                  0.3270
Time:                        00:27:10   Log-Likelihood:                -323.35
converged:                       True   LL-Null:                       -480.45
Covariance Type:            nonrobust   LLR p-value:                 9.300e-67
=======================================================================================
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept               2.6667      0.262     10.178      0.000       2.153       3.180
Pclass_2[T.True]       -1.3012      0.278     -4.678      0.000      -1.846      -0.756
Pclass_3[T.True]       -2.5724      0.282     -9.138      0.000      -3.124      -2.021
Gender_male[T.True]    -2.5139      0.208    -12.108      0.000      -2.921      -2.107
Age                    -0.5387      0.111     -4.850      0.000      -0.756      -0.321
=======================================================================================
"""

In [89]:
X = df[["Age" , "Pclass_2" , "Pclass_3" , "Gender_male"]]
Y = df["Survived"]

X_train, X_test, y_train , y_test = train_test_split(
    X,Y,
    test_size=0.3
)

In [90]:
model = LogisticRegression()
model.fit(X_train, y_train)

LogisticRegression()

In [91]:
y_pred = model.predict(X_test)

In [92]:
from sklearn.metrics import confusion_matrix, accuracy_score

print(confusion_matrix(y_test, y_pred))
print(accuracy_score(y_test, y_pred))

[[107  28]
 [ 24  55]]
0.7570093457943925


In [93]:
model = LogisticRegression()

k = 10
kfold = KFold (n_splits = k, random_state = 0, shuffle = True)

mse_cv = cross_val_score(model, X, Y, cv=kfold, scoring='accuracy')

print(np.mean(mse_cv))

0.7850156494522692
